<a href="https://colab.research.google.com/github/Zain506/MedCLIP-SAM/blob/main/notebooks/ImageSegmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image Segmentation

**MedCLIP-SAM**
[Paper](https://arxiv.org/pdf/2403.20253)


**gScoreCAM**
[Paper](https://www.google.com/url?q=https%3A%2F%2Fopenaccess.thecvf.com%2Fcontent%2FACCV2022%2Fpapers%2FChen_gScoreCAM_What_objects_is_CLIP_looking_at_ACCV_2022_paper.pdf)

In [2]:
%pip install open-clip-torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount("/content/drive/")

Mounted at /content/drive/


## Load BiomedCLIP

In [4]:
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import open_clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
weights_path = "/content/drive/MyDrive/colab/MedCLIP-SAM/biomedclip_weights.pth"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
tokenizer = open_clip.get_tokenizer("ViT-B-32")
model = model.to(device)

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [5]:
# We are looking for the final attention layer
# We place a hook in it to retrieve
modules = dict(model.named_modules())
print(modules["visual"].transformer.resblocks[10].attn) # For full model, print(modules[""])

MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
)


# Define a method to view the activations and gradients

In [6]:
def online_test(img, text, layer=-2): # Run on 1 data point
    activations = {}

    def forward_hook(module, inp, out):
        out = out[0]
        out.retain_grad() # Save gradient
        activations["attn"] = out # Store

    # 2nd last attention layer: Default to 2nd last
    layer = model.visual.transformer.resblocks[layer].attn
    handle_fwd = layer.register_forward_hook(forward_hook) # Register hook

    # forward
    img_emb = model.encode_image(img)
    text_emb = model.encode_text(text)

    # differentiable loss
    sim = img_emb @ text_emb.T
    loss = -sim.diag().mean() # Temporary loss function

    loss.backward() # Backward pass

    handle_fwd.remove() # Remove hook

    act = activations["attn"] # Retrieve activations
    grad = act.grad # Retrieve gradients

    return act, grad, loss


# Load data to test

In [7]:
from datasets import load_dataset
ds = load_dataset("adishourya/MEDPIX-ClinQA") # Same MedPIX dataset that fine-tuned MedCLIP
train_valid = ds["train"].train_test_split(test_size=0.1)
training = train_valid["train"].select(range(100))
test = train_valid["test"]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-7c3594e1ed5838(…):   0%|          | 0.00/449M [00:00<?, ?B/s]

data/train-00001-of-00007-e752d8ec3b321c(…):   0%|          | 0.00/451M [00:00<?, ?B/s]

data/train-00002-of-00007-45744c622d957d(…):   0%|          | 0.00/456M [00:00<?, ?B/s]

data/train-00003-of-00007-66fa0bbd117bd1(…):   0%|          | 0.00/450M [00:00<?, ?B/s]

data/train-00004-of-00007-4b99b1bf376499(…):   0%|          | 0.00/449M [00:00<?, ?B/s]

data/train-00005-of-00007-65c2321df86fd9(…):   0%|          | 0.00/461M [00:00<?, ?B/s]

data/train-00006-of-00007-86735cb396db03(…):   0%|          | 0.00/447M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20500 [00:00<?, ? examples/s]

In [8]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def getData(i): # Retrieve data and generate initial embeddings
  text = tokenizer(training[i]["question"] + "\n" + training[0]["answer"])
  image = preprocess(training[i]["image_id"]).unsqueeze(0).to(device)

  return text, image


In [9]:
text, image = getData(0) # Run data retrieval and tokenisation
maps = online_test(image, text) # Retrieve 2nd last attention layer
final_layer = online_test(image, text, layer=-1) # Retrieve final attention layer to verify CLS token

# Check which token is the CLS token

In [10]:
# print(final_layer[1][0].shape)
# print(final_layer[1][0])
verifier = (final_layer[1][0] == 0).all(dim=1) # This vector checks if every element in the Jacobian (derivative) vector is zero
print(verifier.shape) # 50 vectors each True or False checking if it is filled with zeroes or not
print(verifier) # Only the first one is False, in line with the theory that the first token is the CLS
# The gradients in the final layer are all 0 except the first one
# Therefore we verify the first token is the CLS token and should be removed

torch.Size([50])
tensor([False,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True])


In [31]:
# print(maps[0]) # [0].shape) # Activations
# print(maps[1]) # [0].shape) # Gradients

activations = maps[0][0][1:,] # Remove CLS token from start
gradients = maps[1][0][1:,] # Remove CLS token from start
print(gradients.shape)
mean_grads = gradients.mean(dim=0) # Mean gradients in each "channel"
# print(activations.shape)
# print(gradients)

# Next need to rerank them in descending order
values, indices = torch.sort(mean_grads.abs(), descending=True) # Sort by absolute value (include negative numbers)
print(indices.shape) # Ranking the mean gradients/significance of each of the 768 channels
k = 20
top_k = indices[0:k] # most significant channels across the entire image
print(top_k.shape)
# Next we filter the activations gradient by the top_k tensor
filtered = activations[:,top_k] # Each patch has a k-dimensional vector associated with it
#print(filtered.shape)

torch.Size([49, 768])
torch.Size([768])
torch.Size([20])


In [37]:
image = training[0]["image_id"]
height, width = image.size
print(height)
print(width)

512
512
